In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.retail.silver_sales")

# Gold 1: sales by region
by_region = (
    silver.groupBy("region")
    .agg(F.countDistinct("order_id").alias("orders"),
         F.sum("quantity").alias("units_sold"),
         F.round(F.sum("total_amount"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
)
by_region.write.mode("overwrite").saveAsTable("workspace.retail.gold_sales_by_region")

# Gold 2: sales by product
by_product = (
    silver.groupBy("product_id", "product_name")
    .agg(F.sum("quantity").alias("units_sold"),
         F.round(F.sum("total_amount"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
)
by_product.write.mode("overwrite").saveAsTable("workspace.retail.gold_sales_by_product")

# Gold 3: monthly trend
by_month = (
    silver.withColumn("month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("month")
    .agg(F.countDistinct("order_id").alias("orders"),
         F.round(F.sum("total_amount"), 2).alias("revenue"))
    .orderBy("month")
)
by_month.write.mode("overwrite").saveAsTable("workspace.retail.gold_sales_by_month")

display(by_region)
display(by_product)
display(by_month)

region,orders,units_sold,revenue
West,483,1456,479925.0
South,495,1506,464920.0
North,461,1339,438360.0
East,420,1213,394950.0


product_id,product_name,units_sold,revenue
P001,Laptop,983,884700.0
P002,Phone,881,528600.0
P004,Monitor,928,232000.0
P003,Headphones,851,68080.0
P005,Keyboard,900,40500.0
P006,Mouse,971,24275.0


month,orders,revenue
2026-01,160,164165.0
2026-02,191,176275.0
2026-03,168,177120.0
2026-04,182,186790.0
2026-05,187,191190.0
2026-06,194,164015.0
2026-07,190,157395.0
2026-08,199,169550.0
2026-09,199,204350.0
2026-10,189,187305.0
